# 🎧 Calming Whisper's "Crazy Heads"
### Reproducing a result from the *Calm-Whisper* paper (arXiv:2505.12969)

OpenAI's **Whisper** is one of the best speech-to-text models in the world. But it has a
strange failure: feed it audio with **no speech at all** (a dog barking, a drill, traffic)
and it will often confidently **invent** words. This is called **hallucination**.

The paper you were given (`2505.12969v1.pdf`) makes a remarkable claim: a few specific
**attention heads** inside Whisper's decoder are responsible for most of this hallucination.
**Switch those heads off** and hallucination drops a lot — *without* hurting normal
transcription.

In this notebook you will:
1. Understand **why** transformers have many attention heads, and what **masking** one means.
2. Measure Whisper's **baseline** hallucination rate (on non-speech) and accuracy (WER, on speech).
3. **Mask attention heads** yourself and try to reproduce — or improve on — the paper's result.

> 📄 Keep the paper open. You'll be asked to look up definitions in it.


## 1 · Attention, and why there are *many* heads

A transformer understands a sequence by letting each position **attend** to others — it asks
*"which other tokens should I look at to make sense of this one?"*

A single attention can only track **one kind of relationship** at a time. So transformers run
**several attention "heads" in parallel**, each on its own slice of the representation.
Different heads specialize:

- one head may link a **verb to its subject**,
- another connects a **pronoun to the noun** it refers to,
- another simply tracks **word order / position**,
- …and some heads, it turns out, learn **unhelpful or even harmful** habits.

Whisper-large-v3's decoder has **32 layers**, each with **20 heads**. That's a lot of little
specialists — and this assignment is about finding the ones that misbehave.


## 2 · Two kinds of attention in the Whisper *decoder*

Whisper is an **encoder–decoder** model: the **encoder** turns audio into features, the
**decoder** writes the text. Every decoder layer has **two** attention blocks:

| Attention block | What it looks at | Intuition |
|---|---|---|
| **Self-attention** (`self_attn`) | the text generated so far | keeps the sentence grammatical & coherent |
| **Cross-attention** (`encoder_attn`) | the **audio** features from the encoder | this is how the text "listens" to the sound |

🤔 **Think before you code:** if Whisper invents words for audio containing *no speech*, which
block is the more likely culprit — the one that reads the **text**, or the one that reads the
**audio**? Write down your guess; you'll test it later.


## 3 · What does "masking a head" mean?

**Masking a head = switching it off.** We force that one head's output to **zero**, let the
rest of the model run normally, and observe what changes.

This is a classic scientific move: to learn what a part does, **disable it and measure the
effect** — like studying a brain region by watching what changes when it goes quiet. If
switching off head #k makes hallucinations vanish, head #k was helping cause them.

> ⚠️ **Heads-up:** the most "obvious" way to mask a head in the `transformers` library
> (the `decoder_head_mask` argument) **silently does nothing** in recent versions. Getting
> masking to actually work is part of the challenge. If you're stuck, a working example lives
> in **`hints.md`** — but try first!


## Setup
Run the next cells to load the model, the data, and the metric tools.

In [ ]:
# --- Setup: load Whisper-large-v3 ---
import torch, numpy as np, librosa, warnings
warnings.filterwarnings("ignore")
from transformers import WhisperForConditionalGeneration, WhisperProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "openai/whisper-large-v3"

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME, dtype=torch.float16).to(DEVICE).eval()
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

print("Loaded", MODEL_NAME, "on", DEVICE)
print("Decoder:", model.config.decoder_layers, "layers,",
      model.config.decoder_attention_heads, "heads per layer")


## 4 · Metric 1 — Hallucination rate (non-speech audio)

**UrbanSound8K** = 8,732 clips of environmental sounds (dog barks, drills, sirens, AC units).
**None contain speech.** A trustworthy model should output **nothing** (an empty string).
If it outputs *any* text, that clip counts as a **hallucination**.

$$\text{Hallucination rate}=\frac{\#\{\text{clips that produced any text}\}}{\#\{\text{all clips}\}}$$

📄 **Look it up:** open the paper at **Section 2.2, Equation (1)** and read the authors' exact
definition. Notice it's a simple **yes/no per clip**: did text come out, or not?


## 5 · Metric 2 — Word Error Rate (real speech)

Turning off heads might cut hallucinations — but does it **break normal transcription**?
To check, we measure **Word Error Rate (WER)** on real speech (**LibriSpeech**):

$$\text{WER}=\frac{S+D+I}{N}=\frac{\text{substitutions}+\text{deletions}+\text{insertions}}{\text{words in the reference}}$$

Lower is better (0 = perfect). We test on **test-clean** (easy) and **test-other** (harder, noisier).

🎯 **The whole point of the paper:** a good fix should **lower hallucination** while keeping
**WER almost unchanged**. Both numbers matter.


### Load datasets & metric tools

In [ ]:
# --- Load the datasets ---
from datasets import load_dataset

# Non-speech audio (for hallucination). ~8,700 clips.
urban = load_dataset("mteb/urbansound8K", split="train")

# Real speech (for WER). Each config's test split is named "test".
# NOTE: 'other' downloads on first run (~300 MB).
libri_clean = load_dataset("openslr/librispeech_asr", "clean", split="test")
libri_other = load_dataset("openslr/librispeech_asr", "other", split="test")

print(len(urban), "non-speech clips |",
      len(libri_clean), "clean +", len(libri_other), "other speech utterances")

def audio16k(sample):
    a = sample["audio"]; arr = np.asarray(a["array"])
    if a["sampling_rate"] != 16000:
        arr = librosa.resample(arr, orig_sr=a["sampling_rate"], target_sr=16000)
    return arr

# A small, class-balanced sample of non-speech clips (keeps runtime short).
SAMPLE_PER_CLASS = 15                 # raise for more reliable numbers (slower)
_classes = [str(c) for c in urban["class"]]
URBAN_IDX = []
for c in sorted(set(_classes)):
    URBAN_IDX += [i for i, x in enumerate(_classes) if x == c][:SAMPLE_PER_CLASS]
print("Using", len(URBAN_IDX), "non-speech clips for the hallucination metric")


In [ ]:
# --- Given tools: inference + metrics (you don't need to change these) ---
from tqdm.auto import tqdm

@torch.no_grad()
def transcribe_greedy(model, audio, max_steps=20):
    '''Decode greedily starting ONLY from <|startoftranscript|>.
    This lets Whisper choose to stay silent (emit no text) on non-speech — which is
    exactly what we want to detect hallucinations.'''
    feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE, model.dtype)
    enc = model.model.encoder(feats)                       # encode the audio ONCE
    ids = torch.tensor([[model.config.decoder_start_token_id]], device=DEVICE)
    toks = []
    for _ in range(max_steps):                             # then decode step by step
        logits = model(encoder_outputs=enc, decoder_input_ids=ids).logits
        nxt = logits[0, -1].argmax()
        if nxt.item() == model.config.eos_token_id:
            break
        toks.append(nxt.item())
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return processor.tokenizer.decode(toks, skip_special_tokens=True).strip()

@torch.no_grad()
def transcribe_full(model, audio):
    '''Full, normal transcription (model.generate) — used for WER on real speech.'''
    feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE, model.dtype)
    ids = model.generate(feats)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

def hallucination_rate(model, indices, show=0):
    '''Fraction of non-speech clips that produced ANY text.'''
    n_hall = 0
    for k, i in enumerate(tqdm(indices, desc="UrbanSound8K", leave=False)):
        text = transcribe_greedy(model, audio16k(urban[i]))
        n_hall += int(len(text) > 0)
        if k < show:
            print(f"   {str(urban[i]['class']):16s} -> {text!r}")
    return n_hall / len(indices) * 100

def _edit(r, h):                                            # word-level edit distance
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
    return d[len(h)]

def corpus_wer(model, ds, n):
    '''WER (%) over the first n utterances, using Whisper's English text normalizer.'''
    norm = processor.tokenizer.normalize
    errs = words = 0
    for i in tqdm(range(n), desc="WER", leave=False):
        ref = norm(ds[i]["text"]).split()
        hyp = norm(transcribe_full(model, audio16k(ds[i]))).split()
        errs += _edit(ref, hyp); words += max(len(ref), 1)
    return errs / words * 100

print("Tools ready: transcribe_greedy, transcribe_full, hallucination_rate, corpus_wer")


## Baseline
First, measure Whisper **before** any masking. Keep these numbers.

In [ ]:
# --- Baseline: Whisper with NO masking ---
base_hall = hallucination_rate(model, URBAN_IDX, show=5)
print(f"\nBaseline hallucination rate: {base_hall:.1f}%")

WER_N = 40                                    # utterances per set (raise for reliability)
base_wer_clean = corpus_wer(model, libri_clean, WER_N)
base_wer_other = corpus_wer(model, libri_other, WER_N)
print(f"Baseline WER  test-clean={base_wer_clean:.2f}%   test-other={base_wer_other:.2f}%")


## 6 · Task — mask the heads

The paper says the trouble-makers are **decoder heads #1, #6 and #11**, masked in **every**
layer. But it doesn't have to be self-attention — you can mask either block (recall Concept 2):

- `attn_type="self"`  → mask **self-attention** heads
- `attn_type="cross"` → mask **cross-attention** heads

Below is *almost* a working masking function. **Fill in the one missing line** (look for `TODO`)
so that each masked head's values become **zero**. Stuck? See `hints.md`.

---

**📖 How to read the code (it's only a few new ideas):**

A *pre-hook* lets us peek at the numbers flowing through the model **just before** an attention
block finishes, and edit them. `x = args[0]` is that block of numbers. Its shape is
`[batch, seq, 1280]`:

- **batch** — how many audio clips we process at once,
- **seq** — how many tokens (words) generated so far,
- **1280** — the size of each token's vector. The key fact: **`1280 = 20 heads × 64`**, so this
  vector is just the **20 heads' outputs glued side by side**.

The line `x.view(b, s, 20, 64)` *un-glues* them into separate heads. After that,
**`x[:, :, h, :]` is exactly head `h`'s 64 numbers** — so that's the slice you switch off in the
`TODO`.


In [ ]:
def mask_heads(model, heads, attn_type="self"):
    '''Switch OFF the given heads in EVERY decoder layer.
       attn_type: "self" = self-attention, "cross" = cross-attention.
       Returns hook handles -> pass them to remove_masks() to undo.'''
    block    = "self_attn" if attn_type == "self" else "encoder_attn"
    n_head   = model.config.decoder_attention_heads               # 20
    head_dim = model.model.decoder.layers[0].self_attn.head_dim   # 64

    def make_hook(heads):
        def pre_hook(module, args):
            x = args[0]                          # numbers entering the block: [batch, seq, 1280]
            b, s, d = x.shape
            x = x.view(b, s, n_head, head_dim).clone()   # un-glue 1280 into (20 heads, 64 each)
            for h in heads:                      # x[:, :, h, :] is now "head h's 64 numbers"
                # ###### YOUR CODE HERE ######   (hint: set x[:, :, h, :] to 0)
                pass
                # ############################
            return (x.view(b, s, d),) + args[1:]
        return pre_hook

    handles = []
    for layer in model.model.decoder.layers:
        attn = getattr(layer, block)
        handles.append(attn.out_proj.register_forward_pre_hook(make_hook(heads)))
    return handles


In [ ]:
def remove_masks(handles):
    '''Undo masking: detach all the hooks.'''
    for h in handles:
        h.remove()


### 6a · Verify the paper's claim

Mask **self-attention** heads `[1, 6, 11]` and re-measure the hallucination rate.
Did it drop the way the paper says (≈100% → ≈24%)?


In [ ]:
handles = mask_heads(model, [1, 6, 11], attn_type="self")
hall_paper = hallucination_rate(model, URBAN_IDX)
remove_masks(handles)
print(f"self[1,6,11]: {hall_paper:.1f}%   (baseline was {base_hall:.1f}%)")


### 6b · Did it work? Now go hunting 🔍

If self-attention `[1,6,11]` barely changed anything — don't panic, that's a real result!
**Try other options** and find the mask that reduces hallucination the MOST:
- switch `attn_type` to `"cross"`,
- try a **single** head (e.g. `[1]`, `[6]`, `[11]`, …),
- try small combinations.

Remember to `remove_masks(...)` after each try. Fill in `BEST_HEADS` and `BEST_TYPE` when you've decided.


In [ ]:
# Try different masks and compare their hallucination rates. One trial looks like:
#     handles = mask_heads(model, [1], attn_type="cross")
#     print("cross[1]:", hallucination_rate(model, URBAN_IDX))
#     remove_masks(handles)          # <-- always remove before the next trial!

# ###### YOUR CODE HERE ######

# ############################

BEST_HEADS = ...          # <-- fill in the heads that worked best for you
BEST_TYPE  = ...          # <-- "self" or "cross"


### 6c · Does your best mask keep the accuracy?

A reduction in hallucination is only useful if **WER stays low**. Re-measure WER with your
best mask applied, and compare to the baseline.


In [ ]:
handles = mask_heads(model, BEST_HEADS, attn_type=BEST_TYPE)
wer_clean = corpus_wer(model, libri_clean, WER_N)
wer_other = corpus_wer(model, libri_other, WER_N)
remove_masks(handles)
print(f"With mask {BEST_TYPE}{BEST_HEADS}:")
print(f"  hallucination needs re-running above; WER clean={wer_clean:.2f}% (base {base_wer_clean:.2f}%), "
      f"other={wer_other:.2f}% (base {base_wer_other:.2f}%)")


## 7 · Reflection (write short answers)

1. **Your guess vs. reality:** in Concept 2 you guessed which attention block causes
   hallucination. Which one actually reduced it more — **self** or **cross**?
2. Did the paper's **self-attention [1, 6, 11]** work as claimed on your sample?
3. For your **best mask**, did WER on test-clean and test-other stay close to baseline?
   Why does that matter?
4. In one sentence: what do you think those heads were "doing" to the audio?

> 🧩 Stuck on the masking code? Open **`hints.md`** for a complete worked example.
